## 1. Import Libraries and Load Data

In [1]:
import json
import numpy as np
from collections import defaultdict
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage

In [2]:
with open('actual_pr_reviewer_assignments.json', 'r', encoding='utf-8') as f:
    actual_data = json.load(f)

with open('all_pr_reviewer_assignments.json', 'r', encoding='utf-8') as f:
    predicted_data = json.load(f)

print(f"Total PRs in actual data: {len(actual_data['pr_reviewer_assignments'])}")
print(f"Total PRs in predicted data: {len(predicted_data['pr_assignments'])}")

Total PRs in actual data: 36
Total PRs in predicted data: 36


## 2. Data Preparation

In [3]:
actual_reviewers = {}
predicted_reviewers = {}

for pr in actual_data['pr_reviewer_assignments']:
    pr_number = pr['pr_number']
    actual_reviewers[pr_number] = pr['unique_reviewers']

for pr in predicted_data['pr_assignments']:
    pr_number = pr['pr_number']
    
    # Extract top-3 reviewer names from the detailed predictions
    if 'top_3_reviewers' in pr and pr['top_3_reviewers']:
        top_3_names = [reviewer['reviewer_name'] for reviewer in pr['top_3_reviewers']]
        predicted_reviewers[pr_number] = top_3_names
    else:
        # Fallback to assigned_reviewer if top_3_reviewers is not available
        predicted_reviewers[pr_number] = [pr['assigned_reviewer']] if pr.get('assigned_reviewer') else []

common_prs = set(actual_reviewers.keys()) & set(predicted_reviewers.keys())


## 3. Comprehensive Metric Calculation Functions

In [4]:
def calculate_top_k_accuracy(actual, predicted, k=1):
    if not common_prs:
        return 0.0
    
    hits = 0
    total = 0
    
    for pr_num in common_prs:
        actual_revs = set(actual[pr_num])
        pred_revs = predicted[pr_num][:k]
        
        if any(rev in actual_revs for rev in pred_revs):
            hits += 1
        total += 1
    
    return (hits / total) * 100 if total > 0 else 0.0

In [5]:
# ==================== PRECISION@K ====================
def calculate_precision_at_k(actual, predicted, k):
    """
    Precision@k = (# of recommended items @k that are relevant) / k
    """
    actual_set = set(actual)
    pred_k = predicted[:k]
    
    if k == 0:
        return 0.0
    
    relevant_at_k = sum(1 for p in pred_k if p in actual_set)
    return relevant_at_k / k

def calculate_avg_precision_at_k(actual_dict, predicted_dict, k):
    """Calculate average Precision@k across all PRs"""
    if not common_prs:
        return 0.0
    
    precision_scores = []
    for pr_num in common_prs:
        precision = calculate_precision_at_k(
            actual_dict[pr_num],
            predicted_dict[pr_num],
            k
        )
        precision_scores.append(precision)
    
    return np.mean(precision_scores) * 100

In [6]:
# ==================== RECALL@K ====================
def calculate_recall_at_k(actual, predicted, k):
    """
    Recall@k = (# of recommended items @k that are relevant) / (total # of relevant items)
    """
    actual_set = set(actual)
    pred_k = predicted[:k]
    
    if len(actual_set) == 0:
        return 0.0
    
    relevant_at_k = sum(1 for p in pred_k if p in actual_set)
    return relevant_at_k / len(actual_set)

def calculate_avg_recall_at_k(actual_dict, predicted_dict, k):
    """Calculate average Recall@k across all PRs"""
    if not common_prs:
        return 0.0
    
    recall_scores = []
    for pr_num in common_prs:
        recall = calculate_recall_at_k(
            actual_dict[pr_num],
            predicted_dict[pr_num],
            k
        )
        recall_scores.append(recall)
    
    return np.mean(recall_scores) * 100

In [7]:
# ==================== F1-SCORE@K ====================
def calculate_f1_at_k(actual, predicted, k):
    """
    F1@k = 2 * (Precision@k * Recall@k) / (Precision@k + Recall@k)
    """
    precision = calculate_precision_at_k(actual, predicted, k)
    recall = calculate_recall_at_k(actual, predicted, k)
    
    if precision + recall == 0:
        return 0.0
    
    return 2 * (precision * recall) / (precision + recall)

def calculate_avg_f1_at_k(actual_dict, predicted_dict, k):
    """Calculate average F1@k across all PRs"""
    if not common_prs:
        return 0.0
    
    f1_scores = []
    for pr_num in common_prs:
        f1 = calculate_f1_at_k(
            actual_dict[pr_num],
            predicted_dict[pr_num],
            k
        )
        f1_scores.append(f1)
    
    return np.mean(f1_scores) * 100

In [8]:
# ==================== HIT@K ====================
def calculate_hit_at_k(actual, predicted, k):
    """
    Hit@k = 1 if at least one relevant item is in top-k, else 0
    """
    actual_set = set(actual)
    pred_k = predicted[:k]
    
    return 1.0 if any(p in actual_set for p in pred_k) else 0.0

def calculate_avg_hit_at_k(actual_dict, predicted_dict, k):
    """Calculate average Hit@k across all PRs (same as Top-k Accuracy)"""
    if not common_prs:
        return 0.0
    
    hit_scores = []
    for pr_num in common_prs:
        hit = calculate_hit_at_k(
            actual_dict[pr_num],
            predicted_dict[pr_num],
            k
        )
        hit_scores.append(hit)
    
    return np.mean(hit_scores) * 100

In [9]:
def calculate_average_precision_at_k(actual, predicted, k=5):
    actual_set = set(actual)
    pred_k = predicted[:k]
    
    if len(actual_set) == 0:
        return 0.0
    
    score = 0.0
    num_hits = 0.0
    
    for i, p in enumerate(pred_k):
        if p in actual_set:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    
    return score / min(len(actual_set), k)

def calculate_map_at_k(actual, predicted, k=5):
    if not common_prs:
        return 0.0
    
    ap_scores = []
    
    for pr_num in common_prs:
        ap = calculate_average_precision_at_k(
            actual[pr_num], 
            predicted[pr_num], 
            k
        )
        ap_scores.append(ap)
    
    return np.mean(ap_scores) * 100

In [10]:
def calculate_reciprocal_rank(actual, predicted):
    actual_set = set(actual)
    
    for i, p in enumerate(predicted):
        if p in actual_set:
            return 1.0 / (i + 1.0)
    
    return 0.0

def calculate_mrr(actual, predicted):
    if not common_prs:
        return 0.0
    
    rr_scores = []
    
    for pr_num in common_prs:
        rr = calculate_reciprocal_rank(
            actual[pr_num], 
            predicted[pr_num]
        )
        rr_scores.append(rr)
    
    return np.mean(rr_scores)

In [11]:
# ==================== DCG & NDCG ====================
def calculate_dcg_at_k(actual, predicted, k):
    """
    DCG@k = sum(rel_i / log2(i + 1)) for i in range(k)
    rel_i = 1 if predicted[i] is relevant, else 0
    """
    actual_set = set(actual)
    pred_k = predicted[:k]
    
    dcg = 0.0
    for i, p in enumerate(pred_k):
        if p in actual_set:
            # relevance = 1 for relevant items
            dcg += 1.0 / np.log2(i + 2)  # i+2 because log2(1) = 0
    
    return dcg

def calculate_idcg_at_k(actual, k):
    """
    IDCG@k = DCG of ideal ranking (all relevant items first)
    """
    num_relevant = min(len(actual), k)
    idcg = 0.0
    for i in range(num_relevant):
        idcg += 1.0 / np.log2(i + 2)
    
    return idcg

def calculate_ndcg_at_k(actual, predicted, k):
    """
    NDCG@k = DCG@k / IDCG@k
    """
    dcg = calculate_dcg_at_k(actual, predicted, k)
    idcg = calculate_idcg_at_k(actual, k)
    
    if idcg == 0:
        return 0.0
    
    return dcg / idcg

def calculate_avg_dcg_at_k(actual_dict, predicted_dict, k):
    """Calculate average DCG@k across all PRs"""
    if not common_prs:
        return 0.0
    
    dcg_scores = []
    for pr_num in common_prs:
        dcg = calculate_dcg_at_k(
            actual_dict[pr_num],
            predicted_dict[pr_num],
            k
        )
        dcg_scores.append(dcg)
    
    return np.mean(dcg_scores)

def calculate_avg_ndcg_at_k(actual_dict, predicted_dict, k):
    """Calculate average NDCG@k across all PRs"""
    if not common_prs:
        return 0.0
    
    ndcg_scores = []
    for pr_num in common_prs:
        ndcg = calculate_ndcg_at_k(
            actual_dict[pr_num],
            predicted_dict[pr_num],
            k
        )
        ndcg_scores.append(ndcg)
    
    return np.mean(ndcg_scores) * 100

In [12]:
def calculate_stability_metrics(actual, predicted):
    # Get all unique reviewers
    all_actual_reviewers = set()
    all_predicted_reviewers = set()
    
    for pr_num in common_prs:
        all_actual_reviewers.update(actual[pr_num])
        all_predicted_reviewers.update(predicted[pr_num])
    
    # Coverage: How many actual reviewers were predicted at least once
    coverage = len(all_actual_reviewers & all_predicted_reviewers) / len(all_actual_reviewers) if all_actual_reviewers else 0
    
    # Diversity ratio
    diversity_ratio = len(all_predicted_reviewers) / len(all_actual_reviewers) if all_actual_reviewers else 0
    
    # Exact match rate (predicted reviewer exactly matches one of the actual reviewers)
    exact_matches = sum(1 for pr_num in common_prs if set(predicted[pr_num]) & set(actual[pr_num]))
    exact_match_rate = (exact_matches / len(common_prs)) * 100 if common_prs else 0
    
    return {
        'coverage': coverage * 100,
        'diversity_ratio': diversity_ratio,
        'exact_match_rate': exact_match_rate,
        'unique_actual_reviewers': len(all_actual_reviewers),
        'unique_predicted_reviewers': len(all_predicted_reviewers)
    }

## 4. Calculate All Metrics

In [13]:
# ==================== CALCULATE ALL METRICS ====================

# Precision@k
precision_at_1 = calculate_avg_precision_at_k(actual_reviewers, predicted_reviewers, k=1)
precision_at_3 = calculate_avg_precision_at_k(actual_reviewers, predicted_reviewers, k=3)
precision_at_5 = calculate_avg_precision_at_k(actual_reviewers, predicted_reviewers, k=5)
precision_at_10 = calculate_avg_precision_at_k(actual_reviewers, predicted_reviewers, k=10)

# Recall@k
recall_at_1 = calculate_avg_recall_at_k(actual_reviewers, predicted_reviewers, k=1)
recall_at_3 = calculate_avg_recall_at_k(actual_reviewers, predicted_reviewers, k=3)
recall_at_5 = calculate_avg_recall_at_k(actual_reviewers, predicted_reviewers, k=5)
recall_at_10 = calculate_avg_recall_at_k(actual_reviewers, predicted_reviewers, k=10)

# F1-Score@k
f1_at_1 = calculate_avg_f1_at_k(actual_reviewers, predicted_reviewers, k=1)
f1_at_3 = calculate_avg_f1_at_k(actual_reviewers, predicted_reviewers, k=3)
f1_at_5 = calculate_avg_f1_at_k(actual_reviewers, predicted_reviewers, k=5)
f1_at_10 = calculate_avg_f1_at_k(actual_reviewers, predicted_reviewers, k=10)

# Hit@k (Top-k Accuracy)
hit_at_1 = calculate_avg_hit_at_k(actual_reviewers, predicted_reviewers, k=1)
hit_at_3 = calculate_avg_hit_at_k(actual_reviewers, predicted_reviewers, k=3)
hit_at_5 = calculate_avg_hit_at_k(actual_reviewers, predicted_reviewers, k=5)
hit_at_10 = calculate_avg_hit_at_k(actual_reviewers, predicted_reviewers, k=10)

# MRR (Mean Reciprocal Rank)
mrr = calculate_mrr(actual_reviewers, predicted_reviewers)

# MAP@k (Mean Average Precision)
map_at_1 = calculate_map_at_k(actual_reviewers, predicted_reviewers, k=1)
map_at_3 = calculate_map_at_k(actual_reviewers, predicted_reviewers, k=3)
map_at_5 = calculate_map_at_k(actual_reviewers, predicted_reviewers, k=5)
map_at_10 = calculate_map_at_k(actual_reviewers, predicted_reviewers, k=10)

# DCG@k (Discounted Cumulative Gain)
dcg_at_1 = calculate_avg_dcg_at_k(actual_reviewers, predicted_reviewers, k=1)
dcg_at_3 = calculate_avg_dcg_at_k(actual_reviewers, predicted_reviewers, k=3)
dcg_at_5 = calculate_avg_dcg_at_k(actual_reviewers, predicted_reviewers, k=5)
dcg_at_10 = calculate_avg_dcg_at_k(actual_reviewers, predicted_reviewers, k=10)

# NDCG@k (Normalized Discounted Cumulative Gain)
ndcg_at_1 = calculate_avg_ndcg_at_k(actual_reviewers, predicted_reviewers, k=1)
ndcg_at_3 = calculate_avg_ndcg_at_k(actual_reviewers, predicted_reviewers, k=3)
ndcg_at_5 = calculate_avg_ndcg_at_k(actual_reviewers, predicted_reviewers, k=5)
ndcg_at_10 = calculate_avg_ndcg_at_k(actual_reviewers, predicted_reviewers, k=10)

# Stability Metrics
stability = calculate_stability_metrics(actual_reviewers, predicted_reviewers)

print("✅ All metrics calculated successfully!")

✅ All metrics calculated successfully!


In [14]:
print("="*80)
print("PR REVIEWER ASSIGNMENT - COMPREHENSIVE EVALUATION METRICS")
print("="*80)
print(f"\nTotal PRs Evaluated: {len(common_prs)}")
print("\n" + "="*80)

# ==================== PRECISION@K ====================
print("\n📊 PRECISION@K")
print("-"*80)
print(f"Precision@1:  {precision_at_1:.2f}%")
print(f"Precision@3:  {precision_at_3:.2f}%")
print(f"Precision@5:  {precision_at_5:.2f}%")
print(f"Precision@10: {precision_at_10:.2f}%")

# ==================== RECALL@K ====================
print("\n📊 RECALL@K")
print("-"*80)
print(f"Recall@1:  {recall_at_1:.2f}%")
print(f"Recall@3:  {recall_at_3:.2f}%")
print(f"Recall@5:  {recall_at_5:.2f}%")
print(f"Recall@10: {recall_at_10:.2f}%")

# ==================== F1-SCORE@K ====================
print("\n📊 F1-SCORE@K")
print("-"*80)
print(f"F1@1:  {f1_at_1:.2f}%")
print(f"F1@3:  {f1_at_3:.2f}%")
print(f"F1@5:  {f1_at_5:.2f}%")
print(f"F1@10: {f1_at_10:.2f}%")

# ==================== HIT@K (Top-k Accuracy) ====================
print("\n📊 HIT@K (Top-k Accuracy)")
print("-"*80)
print(f"Hit@1:  {hit_at_1:.2f}%")
print(f"Hit@3:  {hit_at_3:.2f}%")
print(f"Hit@5:  {hit_at_5:.2f}%")
print(f"Hit@10: {hit_at_10:.2f}%")

# ==================== MRR ====================
print("\n📊 MEAN RECIPROCAL RANK (MRR)")
print("-"*80)
print(f"MRR:       {mrr:.4f}")
print(f"MRR (%):   {mrr*100:.2f}%")

# ==================== MAP@K ====================
print("\n📊 MEAN AVERAGE PRECISION (MAP@K)")
print("-"*80)
print(f"MAP@1:  {map_at_1:.2f}%")
print(f"MAP@3:  {map_at_3:.2f}%")
print(f"MAP@5:  {map_at_5:.2f}%")
print(f"MAP@10: {map_at_10:.2f}%")

# ==================== DCG@K ====================
print("\n📊 DISCOUNTED CUMULATIVE GAIN (DCG@K)")
print("-"*80)
print(f"DCG@1:  {dcg_at_1:.4f}")
print(f"DCG@3:  {dcg_at_3:.4f}")
print(f"DCG@5:  {dcg_at_5:.4f}")
print(f"DCG@10: {dcg_at_10:.4f}")

# ==================== NDCG@K ====================
print("\n📊 NORMALIZED DISCOUNTED CUMULATIVE GAIN (NDCG@K)")
print("-"*80)
print(f"NDCG@1:  {ndcg_at_1:.2f}%")
print(f"NDCG@3:  {ndcg_at_3:.2f}%")
print(f"NDCG@5:  {ndcg_at_5:.2f}%")
print(f"NDCG@10: {ndcg_at_10:.2f}%")

# ==================== STABILITY METRICS ====================
print("\n📊 STABILITY METRICS")
print("-"*80)
print(f"Coverage (Actual reviewers in predictions): {stability['coverage']:.2f}%")
print(f"Exact Match Rate:                           {stability['exact_match_rate']:.2f}%")
print(f"Diversity Ratio:                            {stability['diversity_ratio']:.2f}")
print(f"Unique Actual Reviewers:                    {stability['unique_actual_reviewers']}")
print(f"Unique Predicted Reviewers:                 {stability['unique_predicted_reviewers']}")

print("\n" + "="*80)
print("✅ Evaluation Complete!")
print("="*80)

PR REVIEWER ASSIGNMENT - COMPREHENSIVE EVALUATION METRICS

Total PRs Evaluated: 36


📊 PRECISION@K
--------------------------------------------------------------------------------
Precision@1:  27.78%
Precision@3:  16.67%
Precision@5:  10.00%
Precision@10: 5.00%

📊 RECALL@K
--------------------------------------------------------------------------------
Recall@1:  20.83%
Recall@3:  36.11%
Recall@5:  36.11%
Recall@10: 36.11%

📊 F1-SCORE@K
--------------------------------------------------------------------------------
F1@1:  23.15%
F1@3:  22.22%
F1@5:  15.34%
F1@10: 8.67%

📊 HIT@K (Top-k Accuracy)
--------------------------------------------------------------------------------
Hit@1:  27.78%
Hit@3:  41.67%
Hit@5:  41.67%
Hit@10: 41.67%

📊 MEAN RECIPROCAL RANK (MRR)
--------------------------------------------------------------------------------
MRR:       0.3333
MRR (%):   33.33%

📊 MEAN AVERAGE PRECISION (MAP@K)
--------------------------------------------------------------------------

In [15]:
# ==================== METRICS SUMMARY TABLE ====================
import pandas as pd

metrics_summary = pd.DataFrame({
    'k': [1, 3, 5, 10],
    'Precision@k': [precision_at_1, precision_at_3, precision_at_5, precision_at_10],
    'Recall@k': [recall_at_1, recall_at_3, recall_at_5, recall_at_10],
    'F1@k': [f1_at_1, f1_at_3, f1_at_5, f1_at_10],
    'Hit@k': [hit_at_1, hit_at_3, hit_at_5, hit_at_10],
    'MAP@k': [map_at_1, map_at_3, map_at_5, map_at_10],
    'NDCG@k': [ndcg_at_1, ndcg_at_3, ndcg_at_5, ndcg_at_10]
})

print("\n" + "="*80)
print("📊 METRICS SUMMARY TABLE")
print("="*80)
print(metrics_summary.to_string(index=False))
print("\n" + "="*80)


📊 METRICS SUMMARY TABLE
 k  Precision@k  Recall@k      F1@k     Hit@k     MAP@k    NDCG@k
 1    27.777778 20.833333 23.148148 27.777778 27.777778 27.777778
 3    16.666667 36.111111 22.222222 41.666667 29.398148 32.085140
 5    10.000000 36.111111 15.343915 41.666667 29.398148 32.085140
10     5.000000 36.111111  8.670034 41.666667 29.398148 32.085140



In [16]:
# ==================== ANALYZE MULTIPLE REVIEWERS HANDLING ====================
print("🔍 ANALYZING MULTIPLE REVIEWERS IN EVALUATION")
print("=" * 80)

# Analyze the distribution of actual reviewers per PR
reviewer_count_distribution = {}
multiple_reviewer_examples = []

for pr_num in common_prs:
    actual_revs = actual_reviewers[pr_num]
    num_reviewers = len(actual_revs)
    
    # Count distribution
    reviewer_count_distribution[num_reviewers] = reviewer_count_distribution.get(num_reviewers, 0) + 1
    
    # Collect examples of PRs with multiple reviewers
    if num_reviewers > 1:
        multiple_reviewer_examples.append({
            'pr_number': pr_num,
            'actual_reviewers': actual_revs,
            'predicted_top3': predicted_reviewers[pr_num][:3],
            'num_actual': num_reviewers
        })

print(f"📊 REVIEWER COUNT DISTRIBUTION:")
total_prs = len(common_prs)
for count in sorted(reviewer_count_distribution.keys()):
    prs = reviewer_count_distribution[count]
    percentage = (prs / total_prs) * 100
    print(f"  {count} reviewer(s): {prs:2d} PRs ({percentage:4.1f}%)")

print(f"\n🎯 PRs WITH MULTIPLE REVIEWERS:")
print(f"Total PRs with multiple reviewers: {len(multiple_reviewer_examples)}")
print(f"Percentage of multi-reviewer PRs: {(len(multiple_reviewer_examples)/total_prs)*100:.1f}%")

# Show examples of how evaluation handles multiple reviewers
print(f"\n📋 EXAMPLES OF MULTIPLE REVIEWER EVALUATION:")
for i, example in enumerate(multiple_reviewer_examples[:5], 1):
    pr_num = example['pr_number']
    actual = set(example['actual_reviewers'])
    predicted = example['predicted_top3']
    
    # Calculate metrics for this specific PR
    matches_in_top3 = [p for p in predicted if p in actual]
    
    print(f"\n  Example {i}: PR #{pr_num}")
    print(f"    Actual reviewers ({len(actual)}): {', '.join(actual)}")
    print(f"    Predicted top-3: {', '.join(predicted)}")
    print(f"    Matches found: {', '.join(matches_in_top3) if matches_in_top3 else 'None'}")
    print(f"    Hit@3: {'✅ YES' if matches_in_top3 else '❌ NO'}")
    
    # Show how precision/recall calculated for this PR
    if len(predicted) > 0:
        precision_at_3 = len(matches_in_top3) / min(3, len(predicted))
        recall_at_3 = len(matches_in_top3) / len(actual)
        print(f"    Precision@3: {precision_at_3:.3f} ({len(matches_in_top3)}/{min(3, len(predicted))})")
        print(f"    Recall@3: {recall_at_3:.3f} ({len(matches_in_top3)}/{len(actual)})")

print("\n" + "=" * 80)

🔍 ANALYZING MULTIPLE REVIEWERS IN EVALUATION
📊 REVIEWER COUNT DISTRIBUTION:
  1 reviewer(s): 20 PRs (55.6%)
  2 reviewer(s): 11 PRs (30.6%)
  3 reviewer(s):  2 PRs ( 5.6%)
  4 reviewer(s):  2 PRs ( 5.6%)
  5 reviewer(s):  1 PRs ( 2.8%)

🎯 PRs WITH MULTIPLE REVIEWERS:
Total PRs with multiple reviewers: 16
Percentage of multi-reviewer PRs: 44.4%

📋 EXAMPLES OF MULTIPLE REVIEWER EVALUATION:

  Example 1: PR #4355
    Actual reviewers (2): alexbrasetvik, jdalton
    Predicted top-3: jdalton, alexbrasetvik, Cassieminkus1
    Matches found: jdalton, alexbrasetvik
    Hit@3: ✅ YES
    Precision@3: 0.667 (2/3)
    Recall@3: 1.000 (2/2)

  Example 2: PR #4627
    Actual reviewers (2): joeattardi, jdalton
    Predicted top-3: jdalton, blikblum, alexbrasetvik
    Matches found: jdalton
    Hit@3: ✅ YES
    Precision@3: 0.333 (1/3)
    Recall@3: 0.500 (1/2)

  Example 3: PR #6035
    Actual reviewers (3): jonchurch, UlisesGascon, falsyvalues
    Predicted top-3: jdalton, arty-name, alesmit
    Mat

In [17]:
# ==================== SAVE METRICS TO JSON ====================
metrics_results = {
    'evaluation_metadata': {
        'total_prs_evaluated': len(common_prs),
        'unique_actual_reviewers': stability['unique_actual_reviewers'],
        'unique_predicted_reviewers': stability['unique_predicted_reviewers']
    },
    'precision_at_k': {
        'precision_at_1': precision_at_1,
        'precision_at_3': precision_at_3,
        'precision_at_5': precision_at_5,
        'precision_at_10': precision_at_10
    },
    'recall_at_k': {
        'recall_at_1': recall_at_1,
        'recall_at_3': recall_at_3,
        'recall_at_5': recall_at_5,
        'recall_at_10': recall_at_10
    },
    'f1_score_at_k': {
        'f1_at_1': f1_at_1,
        'f1_at_3': f1_at_3,
        'f1_at_5': f1_at_5,
        'f1_at_10': f1_at_10
    },
    'hit_rate_at_k': {
        'hit_at_1': hit_at_1,
        'hit_at_3': hit_at_3,
        'hit_at_5': hit_at_5,
        'hit_at_10': hit_at_10
    },
    'mean_reciprocal_rank': {
        'mrr': mrr,
        'mrr_percentage': mrr * 100
    },
    'mean_average_precision_at_k': {
        'map_at_1': map_at_1,
        'map_at_3': map_at_3,
        'map_at_5': map_at_5,
        'map_at_10': map_at_10
    },
    'dcg_at_k': {
        'dcg_at_1': dcg_at_1,
        'dcg_at_3': dcg_at_3,
        'dcg_at_5': dcg_at_5,
        'dcg_at_10': dcg_at_10
    },
    'ndcg_at_k': {
        'ndcg_at_1': ndcg_at_1,
        'ndcg_at_3': ndcg_at_3,
        'ndcg_at_5': ndcg_at_5,
        'ndcg_at_10': ndcg_at_10
    },
    'stability_metrics': {
        'coverage': stability['coverage'],
        'exact_match_rate': stability['exact_match_rate'],
        'diversity_ratio': stability['diversity_ratio']
    }
}

# Save metrics to JSON file
with open('pr_reviewer_evaluation_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_results, f, indent=2, ensure_ascii=False)

print("✅ Metrics saved to 'pr_reviewer_evaluation_metrics.json'")

✅ Metrics saved to 'pr_reviewer_evaluation_metrics.json'


## 6. Detailed Analysis

In [18]:
# Analyze correct and incorrect predictions with Top-K evaluation
correct_predictions = []
incorrect_predictions = []
partial_matches = []

for pr_num in sorted(common_prs):
    actual = set(actual_reviewers[pr_num])
    predicted_top3 = predicted_reviewers[pr_num][:3] if predicted_reviewers[pr_num] else []
    
    # Check how many of top-3 predictions match actual reviewers
    matching_predictions = [p for p in predicted_top3 if p in actual]
    
    if matching_predictions:
        # At least one prediction matches
        correct_predictions.append({
            'pr_number': pr_num,
            'actual': list(actual),
            'predicted_top3': predicted_top3,
            'matching_predictions': matching_predictions,
            'match_position': [i+1 for i, p in enumerate(predicted_top3) if p in actual],
            'num_matches': len(matching_predictions)
        })
    else:
        # No predictions match
        incorrect_predictions.append({
            'pr_number': pr_num,
            'actual': list(actual),
            'predicted_top3': predicted_top3,
            'matching_predictions': [],
            'num_matches': 0
        })

print("="*80)
print("📊 TOP-3 PREDICTION ANALYSIS")
print("="*80)
print(f"Total PRs Evaluated: {len(common_prs)}")
print(f"Successful Matched (At least 1 match in top-3): {len(correct_predictions)}")
print(f"Failed to Match (No matches in top-3): {len(incorrect_predictions)}")
print(f"\nTop-3 Hit Rate: {(len(correct_predictions) / len(common_prs)) * 100:.2f}%")

📊 TOP-3 PREDICTION ANALYSIS
Total PRs Evaluated: 36
Successful Matched (At least 1 match in top-3): 15
Failed to Match (No matches in top-3): 21

Top-3 Hit Rate: 41.67%


## 7. LLM-based Failure Analysis

In [ ]:
import os

def get_pr_details(pr_num):
    """Extract PR details from the loaded data"""
    # Get PR details from predicted data (which has more info)
    for pr in predicted_data['pr_assignments']:
        if pr['pr_number'] == pr_num:
            return {
                'title': pr.get('pr_title', 'N/A'),
                'description': pr.get('pr_requirements', {}).get('description', 'N/A')[:300],  # Limit length
                'labels': pr.get('pr_requirements', {}).get('labels', []),
                'files_changed': pr.get('pr_requirements', {}).get('files_changed', 'N/A'),
                'complexity': pr.get('pr_requirements', {}).get('complexity', 'N/A'),
                'top_3_scores': [
                    f"{r['reviewer_name']} (score: {r.get('match_score', 'N/A')})" 
                    for r in pr.get('top_3_reviewers', [])
                ]
            }
    return None

def analyze_failure_with_llm(failure_info):
    """Use LLM to analyze why a reviewer assignment failed with actual PR context"""
    
    pr_num = failure_info['pr_number']
    actual = failure_info['actual']
    predicted = failure_info['predicted_top3']
    
    # Get actual PR details
    pr_details = get_pr_details(pr_num)
    
    if pr_details:
        prompt = f"""Analyze why this PR reviewer assignment failed to match:

**PR #{pr_num}**
Title: {pr_details['title']}
Description: {pr_details['description']}
Labels: {', '.join(pr_details['labels']) if pr_details['labels'] else 'None'}
Files Changed: {pr_details['files_changed']}
Complexity: {pr_details['complexity']}

**Actual Reviewers Assigned:** {', '.join(actual)}
**Predicted Top 3:** {', '.join(predicted)}

**Top 3 Predictions with Scores:**
{chr(10).join(pr_details['top_3_scores'])}

Based on the PR details above, Provide a concise reason why the actual reviewer was better 
suited than the predicted ones. Focus on the most important factor only. Check the PR properly and avoid generic answers. Also distinguish betwwen the actual and predicted reviewers."""
    
    # Configure environment variables
    os.environ["OPENAI_API_KEY"] = ""
    os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

    llm = ChatOpenAI(
        model="qwen/qwen-2.5-72b-instruct",
        temperature=0.4,  # Slightly higher for more varied responses
        max_tokens=250,
        openai_api_key=os.environ.get("OPENAI_API_KEY"),
        openai_api_base=os.environ.get("OPENAI_API_BASE")
    )

    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        return response.content
    except Exception as e:
        return f"Error: {str(e)}"

In [20]:
print("🤖 LLM-BASED FAILURE ANALYSIS")

if not incorrect_predictions:
    print("\n✅ All matched!")
else:
    # Analyze all failed predictions
    num_to_analyze = len(incorrect_predictions)
    
    for i in range(num_to_analyze):
        failure = incorrect_predictions[i]
        
        print(f"\n📋 Failure #{i+1}")
        print(f"   PR Number: {failure['pr_number']}")
        print(f"   Actual: {', '.join(failure['actual'])}")
        print(f"   Predicted: {', '.join(failure['predicted_top3'])}")
        print(f"\n   🧠 LLM Analysis:")
        
        analysis = analyze_failure_with_llm(failure)
        
        # Format output
        for line in analysis.split('\n'):
            if line.strip():
                print(f"      {line.strip()}")

🤖 LLM-BASED FAILURE ANALYSIS

📋 Failure #1
   PR Number: 4267
   Actual: huntie
   Predicted: Hovakimyan, josephwccheng, sokarax

   🧠 LLM Analysis:
      The actual reviewer, **huntie**, was better suited for this PR compared to the predicted reviewers because **huntie** has a strong history of reviewing and contributing to PRs related to optional parameters and utility functions in the codebase. This specific PR introduces an optional `resolveEmpty` parameter to the `defaultTo` function, which aligns closely with **huntie**'s expertise and past contributions.
      In contrast, the predicted reviewers (Hovakimyan, josephwccheng, sokarax) have not shown a significant focus on this particular area of the codebase, making **huntie** a more appropriate choice for ensuring the PR is reviewed effectively and efficiently.

📋 Failure #2
   PR Number: 4444
   Actual: jdalton, shubhamzanwar
   Predicted: aminya, josephwccheng, Hovakimyan

   🧠 LLM Analysis:
      The most important factor for 